In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold,train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv("Training.csv")
X = df.drop(columns=["at_risk"])
y = df["at_risk"]
df2 = pd.read_csv("Testing.csv") 

In [3]:
X2 = df2.drop(columns=["id_student","at_risk"])
y2 = df2["at_risk"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Cross-validation only on the training set
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [5]:
def tune_model(model, param_grid, X_train, y_train, cv):
    search = RandomizedSearchCV(
        estimator=model,param_distributions=param_grid,n_iter=80,
        scoring="recall",
        cv=cv,random_state=42,
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train)

    print(f"Best CV Recall: {search.best_score_:.4f}")
    print(f"Best Parameters: {search.best_params_}\n")

    return search.best_estimator_


 
def evaluate_model(name, model, X, y):
    print(f"--- {name} ---")
    y_pred = model.predict(X)
    print(classification_report(y, y_pred))
 

In [6]:
 
# Logistic Regression
'''X_scaled = StandardScaler().fit_transform(X)
lr_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
}
print("Tuning Logistic Regression...")
best_lr = tune_model(LogisticRegression(max_iter=1000), lr_params, X_scaled, y, cv)
evaluate_model("Logistic Regression", best_lr, X_scaled, y)
 
# Random Forest
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [ 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
print("Tuning Random Forest...")
best_rf = tune_model(RandomForestClassifier(random_state=42), rf_params, X, y, cv)
evaluate_model("Random Forest", best_rf, X, y)'''
 
# XGBoost
xgb_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.05, 0.3, 0.1, 0.2, 0.4],
    "subsample": [0.7, 0.8, 1.0],
}
print("Tuning XGBoost...")
best_xgb = tune_model(XGBClassifier(eval_metric="logloss", random_state=42), xgb_params, X_train, y_train, cv)
evaluate_model("XGBoost", best_xgb, X_test, y_test)

Tuning XGBoost...
Best CV Recall: 0.8819
Best Parameters: {'subsample': 0.7, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.4}

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      2847
           1       0.95      0.89      0.92      3214

    accuracy                           0.92      6061
   macro avg       0.92      0.92      0.92      6061
weighted avg       0.92      0.92      0.92      6061



In [7]:
 
# Logistic Regression
'''X_scaled = StandardScaler().fit_transform(X)
lr_params = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
}
print("Tuning Logistic Regression...")
best_lr = tune_model(LogisticRegression(max_iter=1000), lr_params, X_scaled, y, cv)
evaluate_model("Logistic Regression", best_lr, X_scaled, y)
 
# Random Forest
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [ 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}
print("Tuning Random Forest...")
best_rf = tune_model(RandomForestClassifier(random_state=42), rf_params, X, y, cv)
evaluate_model("Random Forest", best_rf, X, y)'''
 
# XGBoost
xgb_params = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [3, 5, 7, 9],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5]
}
print("Tuning XGBoost...")
pipeline = ImbPipeline(steps=[('scaler',  StandardScaler()),('smote',   SMOTE(random_state=42)),('model',   XGBClassifier(eval_metric="logloss", random_state=42))])
best_xgb = tune_model(pipeline, xgb_params, X_train, y_train, cv)
evaluate_model("XGBoost", best_xgb, X_test, y_test)


Tuning XGBoost...
Best CV Recall: 0.8813
Best Parameters: {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 9, 'model__learning_rate': 0.3, 'model__colsample_bytree': 1.0}

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      2847
           1       0.95      0.88      0.92      3214

    accuracy                           0.91      6061
   macro avg       0.91      0.92      0.91      6061
weighted avg       0.92      0.91      0.91      6061



In [9]:
evaluate_model("XGBoost", best_xgb, X2, y2)

--- XGBoost ---
              precision    recall  f1-score   support

           0       0.93      0.76      0.83      1152
           1       0.79      0.94      0.86      1140

    accuracy                           0.85      2292
   macro avg       0.86      0.85      0.85      2292
weighted avg       0.86      0.85      0.85      2292



All the parameters have been prefixed with model_ so that it is able to tune the model through the pipline

In [8]:
best_xgb.save_model("xgbopt.json")

AttributeError: 'Pipeline' object has no attribute 'save_model'